In [1]:
import os
import re
import pandas as pd
from itertools import product

data = []
for file in os.listdir('/home/sagemaker-user/data/judged_scenarios/'):
    if file.endswith('.json'):
        match = re.search(r'batch-scenarios-(\d+)-policies-each-(.+)-temp(\d+\.\d+)\.json', file)
        if match:
            complexity = int(match.group(1))
            model = match.group(2).replace('_', ' ')
            temp = float(match.group(3))
            
            if model.startswith('200-'):
                model = model[4:]
                
            with open(f'/home/sagemaker-user/data/judged_scenarios/{file}', 'r') as f:
                count = f.read().count('"scenario-id"')
            
            data.append([model, temp, complexity, count])

df = pd.DataFrame(data, columns=['Model', 'Temperature', 'Scenario Complexity', 'Num Scenarios'])
df = df.sort_values(['Model', 'Temperature', 'Scenario Complexity'])
print(df.to_markdown(index=False))

# Find missing combinations for Temperature 0
temp0_df = df[df['Temperature'] == 0.0]
models = temp0_df['Model'].unique()
complexities = [4, 6, 8, 10]

existing = set(zip(temp0_df['Model'], temp0_df['Scenario Complexity']))
all_combinations = set(product(models, complexities))
missing = all_combinations - existing

print("\n**Missing combinations for Temperature 0.0:**")
for model, complexity in sorted(missing):
    print(f"- {model}, Scenario Complexity {complexity}")


| Model             |   Temperature |   Scenario Complexity |   Num Scenarios |
|:------------------|--------------:|----------------------:|----------------:|
| claude 3 7 sonnet |           0   |                     4 |             200 |
| claude 3 7 sonnet |           0   |                     6 |             200 |
| claude 3 7 sonnet |           0   |                     8 |             200 |
| claude 3 7 sonnet |           0   |                    10 |             196 |
| claude 3 7 sonnet |           0.1 |                     4 |             200 |
| claude 3 7 sonnet |           0.1 |                     6 |             200 |
| claude 3 7 sonnet |           0.1 |                     8 |             200 |
| claude 3 7 sonnet |           0.1 |                    10 |             196 |
| claude 4 sonnet   |           0   |                     4 |             200 |
| claude 4 sonnet   |           0   |                     6 |             200 |
| claude 4 sonnet   |           0   |   

In [2]:
import boto3

def get_bedrock_pricing(region='us-east-1'):
    """Get current Bedrock pricing for models used in notebooks"""
    
    MODELS = [
        'claude-3-7-sonnet-20250219',
        'claude-sonnet-4-20250514', 
        'claude-opus-4-5-20251101',
        'nova-premier-v1',
        'nova-2-lite-v1',
        'mistral-large-3-2412',
        'deepseek-v3-2501'
    ]
    
    pricing = boto3.client('pricing', region_name='us-east-1')
    
    response = pricing.get_products(
        ServiceCode='AmazonBedrock',
        Filters=[
            {'Type': 'TERM_MATCH', 'Field': 'location', 'Value': region}
        ]
    )
    
    costs = {}
    for product in response['PriceList']:
        product_data = eval(product)
        model_id = product_data['product']['attributes'].get('modelId')
        
        if any(model in model_id for model in MODELS if model_id):
            pricing_info = list(product_data['terms']['OnDemand'].values())[0]
            price_dimensions = list(pricing_info['priceDimensions'].values())[0]
            costs[model_id] = float(price_dimensions['pricePerUnit']['USD'])
    
    return costs


In [4]:
def main():
    get_bedrock_pricing (region='us-east-1')

In [5]:
main()

AccessDeniedException: An error occurred (AccessDeniedException) when calling the GetProducts operation: User: arn:aws:sts::183023889407:assumed-role/AmazonSageMakerUserIAMExecutionRole/SageMaker is not authorized to perform: pricing:GetProducts because no identity-based policy allows the pricing:GetProducts action